# Live-Lastfluss: Netz am Ende von Teil 4

Statischer AC-Lastfluss für eine einzelne Stunde. An jedem Bus lässt sich testweise eine Last oder ein Erzeuger ergänzen, und für jede Last, jeden Erzeuger und jeden Heimspeicher gibt es Schieberegler für Wirk- und Blindleistung. Nach jeder Änderung wird sofort neu gerechnet (`pp.runpp`, etwa 0,05 s).

**Startzustand** (oben links, danach „Laden / zurücksetzen“):

| Start | Werte |
|---|---|
| Bemessungswerte | alle Lasten und Erzeuger mit Nennleistung (CIGRE-Werte, Windpark 30 MW, PV 10 MWp, PV-Haushalte mit kWp), Speicher 0 |
| Winter: Höchstlast | Stunde mit der größten Last der Januarwoche |
| Winter: Starkwind | Stunde mit der größten Rückspeisung ins 110-kV-Netz |
| Sommer: Mittag | Stunde mit der größten PV-Leistung der Juliwoche |
| Sommer: Nacht | Stunde mit der kleinsten Last der Juniwoche |

Die Werte der Stunden stammen aus den Referenzläufen der Prüfungsszenarien (`w4_p_ref_…`); fehlen sie, werden sie beim ersten Start gerechnet (einige Sekunden je Woche).

**Bedienung:**
- **Strang:** welches Spannungsprofil oben gezeigt wird (Abgang 1, Abgang 2, Wohngebiet bis R15 oder R18, Gewerbe, Industrie).
- **U 110 kV:** Spannung des überlagerten Netzes (Referenz 1,03 pu).
- **S1–S3:** Trennschalter schließen.
- **rONT automatisch:** Die drei Ortsnetztrafos stufen wie in Teil 3b (±2 Stufen à 2,5 %, Ziel 0,95 … 1,05 pu im eigenen Teilnetz). Ausgeschaltet lassen sich die Stufen von Hand setzen (Stufe + senkt, Stufe − hebt an).
- **Element ergänzen:** Bus wählen, Last oder Erzeuger, P und Q eingeben, „Element hinzufügen“. Das Element erscheint unter „Eigene Elemente“ mit eigenen Reglern.
- **Regler:** Wirkleistung P und Blindleistung Q je Element; die Regler rechnen beim Loslassen.

**Vorzeichen** wie in pandapower (Verbraucherzählpfeilsystem): Last P > 0 Bezug, Q > 0 induktiv; Erzeuger P > 0 Einspeisung, Q > 0 übererregt (gibt Blindleistung ab), Q < 0 untererregt (nimmt auf); Speicher P > 0 Laden.

**Anzeige:** Kennzahlen (grüner Rand: alles im Arbeitsband und unter 100 %, sonst rot), Spannungsprofil des Strangs mit Werten an jedem Bus (grau gestrichelt: Ausgangszustand nach dem Laden), Spannung an allen Bussen (Strich: Ausgangszustand) und Auslastung der Leitungen und Transformatoren.

Voraussetzung: `ipywidgets` ist installiert (`pip install ipywidgets`) und die Simulation `w4_kombi_W1_P12` aus Teil 4a ist vorhanden. In VS Code und JupyterLab funktionieren die Regler direkt; in einer exportierten HTML-Datei sind sie nicht bedienbar.

## Oberfläche

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # Module liegen in src, das Notebook in notebooks

import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)
%matplotlib inline

import simulation as sim
import live

sim.ONLY_RUN_IF_CHANGED = True
netze = live.startpunkte()          # Startzustände (Bemessungswerte und typische Stunden)
live.app(netze, strang="Wohngebiet")

## Ohne Oberfläche: dieselbe Rechnung im Code

Für Notizen oder wenn die Regler nicht verfügbar sind. Beispiel: Winterabend, zusätzlich 100 kW Last am Strangende R15 (z. B. mehrere Wallboxen):

In [ ]:
import pandas as pd

n = live.LiveNetz(netze[[k for k in netze if k.startswith("Winter: Höchstlast")][0]])
n.rechne(ront=True)
vorher = pd.Series(n.net.res_bus["vm_pu"].to_numpy(), index=n.net.bus["name"])

n.element_hinzufuegen("LV Bus R15", "Last", p_mw=0.10, q_mvar=0.03)
n.rechne(ront=True)
display(pd.Series(n.zusammenfassung()))
n.abbildung("Wohngebiet", ref=vorher)

Weitere Möglichkeiten im Code:

```python
n.net.sgen.loc[n.net.sgen["name"] == "Windpark", "p_mw"] = 20       # Leistung eines Elements setzen
n.net.sgen.loc[n.net.sgen["name"] == "Windpark", "q_mvar"] = -6     # untererregt
n.net.switch.loc[n.net.switch["name"] == "S1", "closed"] = True     # Schalter
n.net.ext_grid["vm_pu"] = 1.00                                      # 110-kV-Spannung
n.rechne(ront=True); n.abbildung("Abgang 1")
```